# Load packages

In [1]:
import psutil

# Get memory information
memoryInfo = psutil.virtual_memory()
total_memory = memoryInfo.total / (1024 ** 3)
available_memory = memoryInfo.available / (1024 ** 3)
used_memory = memoryInfo.used / (1024 ** 3)

print(f"Total Memory: {total_memory:.2f} GB")
print(f"Available Memory: {available_memory:.2f} GB")
print(f"Used Memory: {used_memory:.2f} GB")

Total Memory: 176.89 GB
Available Memory: 76.30 GB
Used Memory: 96.99 GB


In [2]:
import papermill as pm
from pathlib import Path
import shutil

# 0 - Run `generate_nrrd_volumes` function

In [ ]:
import shutil


def _run_tif2nrrd_notebook(
    sample_dir,
    notebook_name,
    tiff_subdir=None,
):
    """
    Internal helper to execute a tif2nrrd notebook using the default
    slices_<sample_name> structure unless overridden.
    """
    sample_dir_path = Path(sample_dir)
    sample_name = sample_dir_path.stem

    if tiff_subdir is None:
        tiff_subdir = f"slices_{sample_name}"

    tiff_path = sample_dir_path / tiff_subdir

    paper_dict = dict(
        image_directory_name=str(tiff_path),
        output_dir_path=str(sample_dir_path / "microCT_volume"),
        pca_directory_name=str(sample_dir_path),
    )

    input_nb = notebook_name
    output_nb = str(sample_dir_path / notebook_name)

    print(input_nb)
    pm.execute_notebook(
        input_nb,
        output_nb,
        parameters=paper_dict,
    )


def generate_nrrd_volume_preview(
    sample_dir,
    tiff_subdir=None,
):
    """
    Generate preview quality NRRD volumes.
    """
    _run_tif2nrrd_notebook(
        sample_dir=sample_dir,
        notebook_name="00_tif2nrrd_previewQuality.ipynb",
        tiff_subdir=tiff_subdir,
    )


def generate_nrrd_volume_full(
    sample_dir,
    tiff_subdir=None,
    delete_tiff=False,
):
    """
    Generate full quality NRRD volumes and optionally delete TIFF files.
    """
    sample_dir_path = Path(sample_dir)
    sample_name = sample_dir_path.stem

    if tiff_subdir is None:
        tiff_subdir = f"slices_{sample_name}"

    tiff_path = sample_dir_path / tiff_subdir

    _run_tif2nrrd_notebook(
        sample_dir=sample_dir,
        notebook_name="00_tif2nrrd_fullQuality.ipynb",
        tiff_subdir=tiff_subdir,
    )

    if delete_tiff:
        try:
            shutil.rmtree(tiff_path)
            print(f"Deleted TIFF directory: {tiff_path}")
        except Exception as e:
            print(f"Error deleting TIFF directory {tiff_path}: {e}")


def generate_nrrd_volumes(
    sample_dir,
    tiff_subdir=None,
    delete_tiff=False,
):
    """
    Run preview and full quality generation using the standard sample layout.
    """
    generate_nrrd_volume_preview(
        sample_dir,
        tiff_subdir=tiff_subdir,
    )

    generate_nrrd_volume_full(
        sample_dir,
        tiff_subdir=tiff_subdir,
        delete_tiff=delete_tiff,
    )


In [ ]:
sample_dir = '/config/researcher_home/Documents/microCT/2025-12-17_GN009/104574/'

# generate_nrrd_volume_preview(sample_dir)
# generate_nrrd_volume_full(sample_dir, delete_tiff=True)
generate_nrrd_volumes(sample_dir, delete_tiff=True)

# 1 - Run `thresholdInspector` function

In [5]:
def thresholdInspector(sample_dir, volume_file = 'microCT_volume/microCT_volume.nrrd'):
    
    sample_path = Path(sample_dir)

    volume_file_path = sample_path / volume_file
    output_dir = sample_path / 'segmented_volumes'

    paper_dict = dict(volume_file=str(volume_file_path),
                  output_dir_path=str(output_dir))

    input_nb = '01_inspect_thresholds.ipynb'
    output_nb =  sample_path / input_nb

    print(input_nb)
    pm.execute_notebook(
       input_nb,
       str(output_nb),
       parameters=dict(paper_dict)
    );

In [13]:
sample_dir = '/config/researcher_home/Documents/microCT/2025-12-17_GN009/104574/'

# thresholdInspector(sample_dir, volume_file = 'microCT_volume/microCT_volume_preview.nrrd')
thresholdInspector(sample_dir)

01_inspect_thresholds.ipynb


Executing: 100%|##########| 48/48 [11:05<00:00, 13.87s/cell]


# 1 - Run `01_set_VOI_calibration` notebook

<span style="color:red; font-size:200%">Manual Task</span>

Run the script manually to select one of the calibration spheres.

In [15]:
sample_dir = '/config/Downloads/104575/'

input_nb = '01_set_VOI_calibration.ipynb'
output_nb =  Path(sample_dir) / input_nb

shutil.copy(input_nb, output_nb)

PosixPath('/config/Downloads/104575/01_set_VOI_calibration.ipynb')

# 1.1 - Run `setCalibrationThreshold` function

Run it only after running `01_set_VOI_calibration.ipynb`, which most likely needs to be done manually.

In [16]:
def setCalibrationThreshold(sample_dir):
    
    sample_path = Path(sample_dir)
    
    output_dir = sample_path / 'microCT_calibration'
    calibration_file = output_dir / 'calibration_stats.csv'
    output_greyvalues_file = sample_path / 'segmented_volumes/segments_greyvalues.csv'

    paper_dict = dict(output_dir = str(output_dir),
                      calibration_file = str(calibration_file),
                      output_greyvalues_file = str(output_greyvalues_file))

    input_nb = '01-1_set_calibration_threshold.ipynb'
    output_nb =  sample_path / input_nb

    print(input_nb)
    pm.execute_notebook(
       input_nb,
       str(output_nb),
       kernel_name='python3',
       parameters=dict(paper_dict)
    );

In [18]:
sample_dir = '/config/Downloads/104575/'

setCalibrationThreshold(sample_dir)

01-1_set_calibration_threshold.ipynb


Executing: 100%|##########| 25/25 [00:03<00:00,  7.56cell/s]


# 2 - Run `microCT_analyzer` function

In [6]:
def microCT_analyzer(sample_dir, volume_file = 'microCT_volume/microCT_volume.nrrd'):  

    sample_path = Path(sample_dir)

    volume_file_path = sample_path / volume_file
    output_dir = sample_path / 'segmented_volumes'
    segments_greyvalues_file_path = output_dir / 'segments_greyvalues.csv'
    segmentMask_inclusion_file = sample_path / 'Mask.seg.nrrd'
    segmentMask_exclusion_file = sample_path / 'microCT_calibration/spheres_calibration.seg.nrrd'
    
    # 02_segment_microCT
    paper_dict = dict(volume_file = str(volume_file_path),
                      output_dir_path = str(output_dir),
                      segments_greyvalues_file = str(segments_greyvalues_file_path),
                      segmentMask_inclusion_file = str(segmentMask_inclusion_file),
                      segmentMask_exclusion_file = str(segmentMask_exclusion_file),
                 )

    input_nb = '02_segment_microCT.ipynb'
    output_nb =  sample_dir + input_nb

    print(input_nb)
    pm.execute_notebook(
       input_nb,
       output_nb,
       parameters=dict(paper_dict)
    );

In [19]:
sample_dir = '/config/Downloads/104575/'

microCT_analyzer(sample_dir, volume_file = 'microCT_volume/microCT_volume_preview.nrrd')

02_segment_microCT.ipynb


Executing: 100%|##########| 54/54 [00:49<00:00,  1.08cell/s]


# 3 - Run `segment2stl` function

In [20]:
def segment2stl(sample_dir, volume_file = 'microCT_volume/microCT_volume.nrrd'):  

    sample_path = Path(sample_dir)

    volume_file_path = sample_path / volume_file
    segment_file = sample_path / 'segmented_volumes/Bone.seg.nrrd'
    output_dir_path = sample_path / 'segmented_volumes'
    
    # 02_segment_microCT
    paper_dict = dict(volume_file = str(volume_file_path),
                      segment_file = str(segment_file),
                      output_dir_path = str(output_dir_path)                 )

    input_nb = '03_segment2stl.ipynb'
    output_nb =  sample_dir + input_nb

    print(input_nb)
    pm.execute_notebook(
       input_nb,
       output_nb,
       parameters=dict(paper_dict)
    );

In [ ]:
sample_dir = '/config/Downloads/104575/'

segment2stl(sample_dir, volume_file = 'microCT_volume/microCT_volume_preview.nrrd')

03_segment2stl.ipynb


Executing:  85%|########5 | 47/55 [00:42<00:05,  1.51cell/s]

# 4 - Run `04_set_ROI_mineralization` notebook

<span style="color:red; font-size:200%">Manual Task</span>

Run the script manually to position the ROI object

In [ ]:
sample_dir = '/config/Downloads/104575/'

input_nb = '04_set_ROI_mineralization.ipynb'
output_nb =  Path(sample_dir) / input_nb

shutil.copy(input_nb, output_nb)

# 5 - Run `roiStatistics` function

In [3]:
def roiStatistics(sample_dir, volume_file = 'microCT_volume/microCT_volume.nrrd'):  

    sample_path = Path(sample_dir)

    volume_file_path = sample_path / volume_file
    ROI_dir = sample_path / 'segmented_volumes/ROI/'
    ROI_files=['CallusCylinder.stl', 'FemurCylinder.stl', 'OutsideCylinder.stl']
    file_segmentation = sample_path / 'segmented_volumes/Bone.seg.nrrd'
    directory_notebook = sample_path
    sample_name = directory_notebook.stem
    
    # 02_segment_microCT
    paper_dict = dict(volume_file = str(volume_file_path),
                      ROI_dir = str(ROI_dir),
                      ROI_files = ROI_files,
                      file_segmentation = str(file_segmentation),
                      directory_notebook = str(directory_notebook),
                      sample_name = sample_name
                      )

    input_nb = '05_ROI_statistics.ipynb'
    output_nb =  sample_dir + input_nb

    print(input_nb)
    pm.execute_notebook(
       input_nb,
       output_nb,
       parameters=dict(paper_dict)
    );

In [5]:
sample_dir = '/config/Downloads/104574/'

roiStatistics(sample_dir, volume_file = 'segmented_volumes/Bone.seg.nrrd')

05_ROI_statistics.ipynb


Executing: 100%|##########| 50/50 [00:44<00:00,  1.11cell/s]


# 6 - Run `visualizer3D` function

In [13]:
def visualizer3D(sample_dir):  

    sample_path = Path(sample_dir)

    model_filename = sample_path / 'segmented_volumes/Bone.stl'

    ROI_dir = sample_path / 'segmented_volumes/ROI/'
    ROI_files=['CallusCylinder.stl', 'FemurCylinder.stl', 'OutsideCylinder.stl']
    
    # 02_segment_microCT
    paper_dict = dict(model_dir = str(sample_path),
                      model_filename = str(model_filename),
                      ROI_dir = str(ROI_dir),
                      ROI_files = ROI_files,
                      )

    input_nb = '06_3Dvisualizer.ipynb'
    output_nb =  sample_dir + input_nb

    print(input_nb)
    pm.execute_notebook(
       input_nb,
       output_nb,
       parameters=dict(paper_dict)
    );

In [14]:
sample_dir = '/config/Downloads/104574/'

visualizer3D(sample_dir)

06_3Dvisualizer.ipynb


Executing: 100%|##########| 20/20 [03:32<00:00, 10.63s/cell]
